## Gold Layer — Driver Career Statistics
Joins `drivers_silver` + `results_silver` to produce an analytics-ready table with career-level KPIs per driver.

In [0]:
df_drivers = spark.table("f1_catalognew.silver_schema.drivers_silver")
df_results = spark.table("f1_catalognew.silver_schema.results_silver")

print(f"Drivers: {df_drivers.count()} rows | Results: {df_results.count()} rows")

In [0]:
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, min as _min, max as _max,
    when, round as _round, current_timestamp
)

# Aggregate race results per driver
df_results_agg = (
    df_results
    .groupBy("driver_id")
    .agg(
        count("result_id").alias("total_races"),
        _round(_sum("points"), 1).alias("total_points"),
        _sum(when(col("position") == 1, 1).otherwise(0)).alias("total_wins"),
        _sum(when(col("position").isin(1, 2, 3), 1).otherwise(0)).alias("total_podiums"),
        _round(avg("position_order"), 2).alias("avg_finish_position"),
        _min("position_order").alias("best_finish"),
        _sum("laps").alias("total_laps"),
        _round(_max("fastest_lap_speed"), 3).alias("top_fastest_lap_speed"),
    )
)

# Join with driver dimensions
df_gold = (
    df_drivers
    .select("driver_id", "driver_ref", "code", "full_name",
            "nationality", "date_of_birth")
    .join(df_results_agg, on="driver_id", how="inner")
    .withColumn("avg_points_per_race",
                _round(col("total_points") / col("total_races"), 2))
    .withColumn("win_rate_pct",
                _round((col("total_wins") / col("total_races")) * 100, 2))
    .withColumn("ingestion_timestamp", current_timestamp())
    .select(
        "driver_id", "driver_ref", "code", "full_name",
        "nationality", "date_of_birth",
        "total_races", "total_points", "total_wins", "total_podiums",
        "win_rate_pct", "avg_points_per_race", "avg_finish_position",
        "best_finish", "total_laps", "top_fastest_lap_speed",
        "ingestion_timestamp"
    )
    .orderBy(col("total_points").desc())
)

display(df_gold)

In [0]:
gold_table = "f1_catalognew.gold_schema.driver_career_stats"
gold_location = "abfss://goldlayer@f1datalakes2026.dfs.core.windows.net/"

# Create gold schema with external managed location
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS f1_catalognew.gold_schema
    MANAGED LOCATION '{gold_location}'
""")

(
    df_gold
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"✓ Gold table written: {gold_table}")
print(f"  Storage: {gold_location}")

In [0]:
df_validated = spark.table("f1_catalognew.gold_schema.driver_career_stats")

print(f"Total drivers in gold: {df_validated.count()}")
print(f"\nTop 10 drivers by career points:")
display(
    df_validated
    .select("full_name", "nationality", "total_races", "total_points",
            "total_wins", "total_podiums", "win_rate_pct", "avg_points_per_race")
    .orderBy(col("total_points").desc())
    .limit(10)
)